# Практика · Аналіз тональності в бою

> Теорія: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

Цей зошит — закінчений приклад того, як **чесно заміряти** класифікатор тексту,
який мають поставити в роботу. Ми навчимо найпростішу модель і далі цілий зошит
будемо дивитися не на модель, а **на число, яким про неї звітують**.

Що зробимо:

1. зберемо корпус із повідомлень програм, установлених на цій машині;
2. подивимось, **чи правило розмітки взагалі каже те, що ми думаємо** — і знайдемо
   в ньому дірку;
3. поставимо два рубежі: «завжди збій» і список ключових слів;
4. навчимо модель і заміряємо її **двома різними розбиттями**, по пʼять зерен;
5. розкладемо якість **по доменах** — і побачимо, що більшість зрізів надто малі,
   щоб мати думку;
6. зробимо **нульовий тест**: перевіримо, чи найгірший зріз гірший за те, що дає
   сам лише шум;
7. подивимось, скільки коштує **поріг**.

⏱ **Заміряно: близько хвилини процесорного часу** на чотирьох ядрах без
відеокарти — у трьох наших прогонах виходило 61, 63 і 66 секунд.
Зошит навчає модель шістнадцять разів, і це і є предмет теми. За годинником вийде
помітно довше — на нашій машині до чотирьох хвилин, — бо поруч рахувалось інше.
Тому останньою клітинкою зошит друкує саме **процесорний** час: годинник на
завантаженій машині не міряє нічого.

## ⚠️ Спершу — чесно про дані

Розміченого корпусу тональності українською, який можна взяти **офлайн**, не існує.
Тому ми беремо задачу **тієї самої форми** на справжніх даних.

У системі стоять сотні програм, і майже кожна перекладена українською. Поруч із
кожним українським рядком лежить **англійський оригінал**. Мітку ми беремо з
оригіналу — чи повідомлення про **збій**, чи про **успіх**, — а класифікуємо
**український переклад**.

**Чому це чесно, наскільки взагалі можливо:** мітку писала людина — розробник, який
знав, що сталося, — і писав її, не думаючи про наш замір. Мітка живе в англійському
рядку, ознаки — в українському; це два різні тексти, тому замір не круговий.

**І чому це все одно проксі:** прагматична функція повідомлення — **не тональність**.
Повідомлення про збій не «зневажливе», воно просто повідомляє про збій. Ми взяли
задачу з тією самою формою — два незбалансовані класи, багато доменів, мітка від
людини, — щоб на ній були видні всі ефекти. Але жодне число нижче **не є** числом
«якості аналізу тональності українською». Це борг курсу, а не досягнення.

**І ще:** корпус збирається з того, що встановлено **на твоїй машині**. Набір програм
у кожного свій, тому твої числа не збігатимуться з тими, що в лекції. Відтворюється
не число, а **форма**: сильний дисбаланс, розрив між двома розбиттями і те, що
більшість доменів надто малі для заміру.

In [ ]:
# Потоки фіксуємо ДО імпорту numpy: інакше очікування потоків OpenMP
# рахується як робота, і процесорний час роздувається в десятки разів.
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[_v] = '1'

import sys, glob, gettext, re, random, collections, statistics, time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support

START_CPU = time.process_time()
print('Python     ', sys.version.split()[0])
print('numpy      ', np.__version__)
import sklearn; print('scikit-learn', sklearn.__version__)

## 1 · Чи є взагалі дані

Корпус — це файли перекладів `.mo`, які лежать у системі. Якщо їх немає (наприклад,
українська локаль не встановлена), далі йти немає сенсу — і краще сказати про це
людською мовою, ніж упасти стеком на двадцятій клітинці.

In [ ]:
LOCALE_DIR = '/usr/share/locale/uk/LC_MESSAGES'
mo_files = sorted(glob.glob(os.path.join(LOCALE_DIR, '*.mo')))

if not mo_files:
    print('Українських перекладів у системі не знайдено.')
    print(f'Очікувалась тека {LOCALE_DIR} з файлами *.mo.')
    print()
    print('Це не помилка зошита — це означає, що на цій машині немає української')
    print('локалі. У Fedora її ставлять пакетом langpacks-uk, у Debian та Ubuntu —')
    print('через dpkg-reconfigure locales (варіант uk_UA.UTF-8).')
    print('Шлях для Debian ми НЕ перевіряли: у нас Debian немає.')
else:
    print(f'Знайдено файлів перекладу: {len(mo_files)}')
    print('Перші пʼять:', [os.path.basename(p) for p in mo_files[:5]])

## 2 · Читаємо паралельні пари «оригінал → переклад»

Усередині `.mo` лежить словник: ключ — англійський рядок, значення — український.
Модуль `gettext` уміє його читати. Беремо лише рядки, довші за 10 символів: коротші
— це здебільшого окремі слова з меню, у яких немає ані збою, ані успіху.

In [ ]:
def load_messages(min_len=10):
    """Повертає [(програма, англійський оригінал, український переклад)]."""
    out = []
    for path in mo_files:
        program = os.path.basename(path)[:-3]          # ім'я файлу без .mo — це програма
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)._catalog
        except Exception:
            continue                                    # трапляються биті або чужі формати
        for source, target in catalog.items():
            if not isinstance(source, str) or not isinstance(target, str):
                continue
            # службовий заголовок каталогу — не повідомлення
            if len(target) > min_len and 'Project-Id' not in target:
                out.append((program, source, target))
    return out

messages = load_messages()
print(f'усього паралельних пар: {len(messages)}')
print(f'програм: {len({m[0] for m in messages})}')
print()
print('приклад:')
program, source, target = messages[0]
print(f'  програма  {program}')
print(f'  оригінал  {source[:70]}')
print(f'  переклад  {target[:70]}')

## 3 · Ставимо мітку за англійським оригіналом

Правило просте: якщо в оригіналі є маркер збою (`error`, `failed`, `cannot`…) —
мітка «збій». Якщо маркер успіху (`done`, `installed`, `saved`…) — мітка «успіх».
Рядки, де є **обидва** маркери або **жодного**, ми викидаємо: там правило не має
однозначної думки, і нав'язувати її не варто.

In [ ]:
FAIL_WORDS = re.compile(
    r"\b(error|failed|failure|cannot|can't|unable|invalid|denied|"
    r"not found|no such|missing|corrupt|abort|fatal|refus\w*|wrong|bad|"
    r"too (many|few|large|long))\b", re.I)
OK_WORDS = re.compile(
    r"\b(success\w*|complete[d]?|done|saved|created|installed|finished|"
    r"ready|ok\b|added|connected|enabled|loaded|updated)\b", re.I)

FAIL, OK = 0, 1          # мітки; 0 — більший клас, 1 — менший

rows = []
for program, source, target in messages:
    is_fail = bool(FAIL_WORDS.search(source))
    is_ok = bool(OK_WORDS.search(source))
    if is_fail == is_ok:          # обидва маркери або жодного — пропускаємо
        continue
    rows.append((program, target, source, FAIL if is_fail else OK))

programs = sorted({r[0] for r in rows})
counts = collections.Counter(r[3] for r in rows)
print(f'рядків {len(rows)} · програм {len(programs)}')
print(f'«збій»  {counts[FAIL]}')
print(f'«успіх» {counts[OK]}')
print(f'частка більшого класу: {counts[FAIL] / len(rows):.4f}')
print(f'співвідношення класів: {counts[FAIL] / counts[OK]:.1f} до 1')

## 4 · Перше, що треба зробити з будь-яким набором даних

Прочитати руками кілька десятків прикладів **меншого** класу й перевірити, чи мітка
справді означає те, що ти думаєш. Зробімо це.

In [ ]:
ok_rows = [r for r in rows if r[3] == OK]
print('десять випадкових рядків із міткою «успіх»:\n')
for program, target, source, label in random.Random(0).sample(ok_rows, 10):
    print(f'  EN  {source[:76]}')
    print(f'  UK  {target[:76]}')
    print()

Уже на десятьох прикладах видно проблему: серед них трапляються рядки на кшталт
«**could not** be loaded». Правило побачило слово `loaded` зі списку успіху, а
заперечення `could not` у списку збою **немає** — там є `cannot`, але не `could not`.

Це не дрібниця, це **систематична помилка розмітки в один бік**. Порахуймо її.

In [ ]:
# заперечення, яких немає в списку маркерів збою
MISSED_NEGATION = re.compile(
    r"\b(could not|couldn't|won't|would not|wouldn't|didn't|did not|"
    r"doesn't|does not|isn't|is not|no longer|never|unsuccessful|"
    r"not (?:be )?(?:possible|supported|allowed|available)|"
    r"nothing|none of|without)\b", re.I)

mislabelled = [r for r in ok_rows if MISSED_NEGATION.search(r[2])]
share = len(mislabelled) / len(ok_rows)
print(f'рядків із міткою «успіх»: {len(ok_rows)}')
print(f'з них насправді описують збій: {len(mislabelled)} ({share:.4f})')
print()
which = collections.Counter(MISSED_NEGATION.search(r[2]).group(0).lower()
                            for r in mislabelled)
print('які саме заперечення проґавлено:')
for word, n in which.most_common(6):
    print(f'  {word:16s} {n}')
print()
print('приклад:')
print(f'  EN  {mislabelled[0][2][:76]}')
print(f'  UK  {mislabelled[0][1][:76]}')

**Що з цього випливає.** Частина меншого класу має **хибну** мітку, і хибну
систематично. Модель, яка правильно розуміє, що «не вдалося завантажити» — це збій,
за це **карається**. Отже справжня стеля цієї задачі нижча за одиницю, і частину
розриву між нашим результатом і одиницею створює не модель, а розмітка.

Ми **не** будемо чинити правило. Причина навчальна: далі в зошиті буде видно, як
шум розмітки поводиться в замірах, а це важливіше за пів пункта якості. Але знати
про дірку — обовʼязково, і саме тому клітинка вище стоїть **до** навчання, а не після.

## 5 · Два рубежі, з якими треба порівнювати

Перш ніж навчати будь-що, треба знати, що дають дві найдурніші системи. Без цього
число моделі ні про що не говорить.

**Рубіж 1 — «завжди збій».** Не дивиться на текст узагалі.
**Рубіж 2 — список ключових слів.** Шукає в **українському** тексті очевидні маркери
збою. Він відповідає на важливе питання: чи не звелася наша задача до того, щоб
упізнати перекладене слово «помилка»?

In [ ]:
y_true = np.array([r[3] for r in rows])

# рубіж 1: на все відповідаємо «збій»
always_fail = np.full_like(y_true, FAIL)

# рубіж 2: список українських маркерів збою
UA_FAIL_WORDS = re.compile(
    r'помилк|не вдал|неможлив|збій|відмовл|не знайден|немає такого|'
    r'пошкодж|недійсн|відсутн|заборон|некоректн|не існує|аварійн', re.I)
by_keyword = np.array([FAIL if UA_FAIL_WORDS.search(r[1]) else OK for r in rows])

for name, pred in (('завжди «збій»', always_fail), ('список слів', by_keyword)):
    acc = accuracy_score(y_true, pred)
    macro = f1_score(y_true, pred, average='macro')
    print(f'{name:16s} accuracy {acc:.4f} · macro-F1 {macro:.4f}')

Зупинись тут на хвилину, бо в цих чотирьох числах — головна думка теми.

Порожня модель має **вищу accuracy**, ніж список ключових слів. Список ключових слів
має **вищу macro-F1**. Дві метрики впорядкували ті самі дві системи **у зворотному
порядку**.

Причина в тому, що знаменник accuracy — усі приклади разом, а їх на 90 % з лишком
складає один клас. Тому accuracy майже цілком міряє, як система обробляє **більший**
клас — і порожня модель тут ідеальна. Macro-F1 усереднює F1 кожного класу з
однаковою вагою, тож нуль на меншому класі сховати не вдається.

Далі всі числа — **macro-F1**.

## 6 · Порахуймо macro-F1 самі, щоб у ньому не було магії

Найкращий спосіб перестати боятись метрики — написати її в пʼять рядків і звірити
з бібліотечною.

In [ ]:
def f1_of_class(y_true_arr, y_pred_arr, target_class):
    """F1 одного класу: гармонічне середнє точності й повноти."""
    predicted_as = (y_pred_arr == target_class)
    really_is = (y_true_arr == target_class)
    hit = int(np.sum(predicted_as & really_is))
    if hit == 0:
        return 0.0
    precision = hit / int(np.sum(predicted_as))     # коли кажемо цей клас — чи маємо рацію
    recall = hit / int(np.sum(really_is))           # яку частку цього класу помітили
    return 2 * precision * recall / (precision + recall)

def macro_f1(y_true_arr, y_pred_arr):
    """Просте середнє F1 по класах: кожен клас важить однаково."""
    return sum(f1_of_class(y_true_arr, y_pred_arr, c) for c in (FAIL, OK)) / 2

our = macro_f1(y_true, by_keyword)
theirs = f1_score(y_true, by_keyword, average='macro')
print(f'наш розрахунок    {our:.6f}')
print(f'бібліотечний      {theirs:.6f}')
assert np.allclose(our, theirs), 'розрахунок розійшовся!'
print('✅ збігається — усередині метрики немає нічого, крім цих пʼятьох рядків')

## 7 · Модель

Навмисно найпростіша: TF-IDF на **символьних** `n`-грамах довжиною 3-5 плюс
логістична регресія. Символьні `n`-грами тут доречні, бо українська сильно
відмінювана: «не вдалося», «не вдається», «не вдалось» — три різні слова й майже
однакові набори символьних шматків.

`class_weight='balanced'` каже моделі важити рідкісний клас сильніше, пропорційно до
його рідкості. Без цього вона просто навчилася б відповідати «збій» на все.

In [ ]:
def train_model(train_rows, seed=0):
    """Повертає пару (векторизатор, модель), навчену на переданих рядках."""
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                                 min_df=2, max_features=50000)
    X = vectorizer.fit_transform([r[1] for r in train_rows])
    model = LogisticRegression(max_iter=1000, class_weight='balanced',
                               random_state=seed)
    model.fit(X, [r[3] for r in train_rows])
    return vectorizer, model

def score_on(vectorizer, model, test_rows):
    """macro-F1 навченої моделі на переданих рядках."""
    predicted = model.predict(vectorizer.transform([r[1] for r in test_rows]))
    return f1_score([r[3] for r in test_rows], predicted, average='macro')

# швидка проба: навчаємось на 80 % рядків, дивимось на решті
shuffled = list(range(len(rows)))
random.Random(0).shuffle(shuffled)
cut = int(0.8 * len(shuffled))
probe_train = [rows[i] for i in shuffled[:cut]]
probe_test = [rows[i] for i in shuffled[cut:]]

t0 = time.process_time()
vec, mdl = train_model(probe_train)
print(f'навчено за {time.process_time() - t0:.1f} с процесорного часу')
print(f'ознак у словнику: {len(vec.vocabulary_)}')
print(f'macro-F1 на відкладеній частині: {score_on(vec, mdl, probe_test):.4f}')

## 8 · Головне рішення зошита: **як** розбивати

Ми щойно перемішали всі рядки й відклали пʼяту частину. Здається чесним. Але
подумай, що при цьому сталося з програмами: рядки **кожної** програми потрапили і в
навчання, і в перевірку. Модель бачила, як пише `git`, і її перевіряють на тому, як
пише `git`.

Таке число відповідає на питання «як модель працюватиме на **нових повідомленнях
програм, які вона вже знає**». Питання законне. Проблема в тому, що обіцяють
зазвичай відповідь на інше: «як вона працюватиме там, де її ще не бачили».

Заміряємо обидва. Пʼять зерен, бо на трьох розкид виходить систематично заниженим —
вони просто не встигають натрапити на невдалий поділ.

In [ ]:
TEST_SHARE = 0.2

def random_split(seed):
    """Рядки перемішані: програми течуть між частинами."""
    order = list(range(len(rows)))
    random.Random(seed).shuffle(order)
    k = int((1 - TEST_SHARE) * len(order))
    return [rows[i] for i in order[:k]], [rows[i] for i in order[k:]]

def by_program_split(seed):
    """Цілі програми: жодного рядка перевірної програми модель не бачить."""
    shuffled_programs = programs[:]
    random.Random(seed).shuffle(shuffled_programs)
    held_out, total = set(), 0
    for program in shuffled_programs:
        held_out.add(program)
        total += sum(1 for r in rows if r[0] == program)
        if total >= TEST_SHARE * len(rows):
            break
    train = [r for r in rows if r[0] not in held_out]
    test = [r for r in rows if r[0] in held_out]
    return train, test

pile_random, pile_domain = [], []
for seed in range(5):
    tr, te = random_split(seed)
    v, m = train_model(tr, seed)
    a = score_on(v, m, te)

    tr, te = by_program_split(seed)
    v, m = train_model(tr, seed)
    b = score_on(v, m, te)

    pile_random.append(a); pile_domain.append(b)
    print(f'зерно {seed}: випадкове {a:.4f} · за програмами {b:.4f}')

print()
for name, pile in (('випадкове   ', pile_random), ('за програмами', pile_domain)):
    print(f'{name} купа {min(pile):.4f}–{max(pile):.4f} · середнє {statistics.mean(pile):.4f}')
print()
overlap = min(pile_random) <= max(pile_domain)
print('купи перекриваються:', overlap)
print('різниця середніх:', f'{statistics.mean(pile_random) - statistics.mean(pile_domain):.4f}')

**Як це читати.** Ми порівнюємо не середні, а **купи**. Якщо найгірше значення
однієї купи краще за найкраще значення другої — різниця є. Якщо купи
перетинаються — різниці не доведено, хоч би як розходились середні.

## 9 · Звідки береться перевага випадкового розбиття

Одна з причин лежить на поверхні: різні програми часто містять **дослівно однакові**
повідомлення («Не вдалося відкрити файл» пишуть усі). При випадковому розбитті такі
рядки опиняються по обидва боки, і модель просто їх памʼятає.

In [ ]:
tr, te = random_split(0)
seen_texts = {r[1] for r in tr}
is_duplicate = np.array([r[1] in seen_texts for r in te])

v, m = train_model(tr, 0)
predicted = m.predict(v.transform([r[1] for r in te]))
y_te = np.array([r[3] for r in te])

print(f'перевірних рядків: {len(te)}')
print(f'дослівно є в навчанні: {int(is_duplicate.sum())} ({is_duplicate.mean():.4f})')
print()
print(f'macro-F1 на дослівних повторах: '
      f'{f1_score(y_te[is_duplicate], predicted[is_duplicate], average="macro"):.4f}')
print(f'macro-F1 на решті:              '
      f'{f1_score(y_te[~is_duplicate], predicted[~is_duplicate], average="macro"):.4f}')

Дослівне запамʼятовування є, але воно пояснює лише частину розриву. Решта — те, що
модель вивчила **манеру** конкретних програм: улюблені звороти, спосіб будувати
речення. У бою на новій програмі цього немає.

## 10 · Груповий пʼятифолд: кожен домен оцінено поза доменом рівно раз

Одного доменного поділу мало: у ньому перевіряються лише ті програми, що випадково
потрапили у відкладену частину. Зробімо так, щоб **кожна** програма побувала поза
навчанням: поділимо 240 програм на пʼять частин і по черзі відкладемо кожну.

Частини набираємо жадібно — найбільші програми першими в найменш завантажену
частину, — щоб фолди вийшли приблизно однакові за обсягом.

In [ ]:
program_size = {p: sum(1 for r in rows if r[0] == p) for p in programs}

folds = [[] for _ in range(5)]
fold_load = [0] * 5
for program in sorted(programs, key=lambda p: -program_size[p]):
    lightest = fold_load.index(min(fold_load))
    folds[lightest].append(program)
    fold_load[lightest] += program_size[program]

# збираємо передбачення по всіх доменах в один довгий список
oof_true, oof_pred, oof_prob, oof_program = [], [], [], []

for j, fold in enumerate(folds):
    held_out = set(fold)
    train = [r for r in rows if r[0] not in held_out]
    test = [r for r in rows if r[0] in held_out]
    vectorizer, model = train_model(train, seed=0)
    X_test = vectorizer.transform([r[1] for r in test])
    predicted = model.predict(X_test)
    oof_pred += list(predicted)
    oof_prob += list(model.predict_proba(X_test)[:, 1])   # ймовірність класу «успіх»
    oof_true += [r[3] for r in test]
    oof_program += [r[0] for r in test]
    print(f'фолд {j}: програм {len(fold):3d} · рядків {len(test):5d} · '
          f'macro-F1 {f1_score([r[3] for r in test], predicted, average="macro"):.4f}')

oof_true = np.array(oof_true)
oof_pred = np.array(oof_pred)
oof_prob = np.array(oof_prob)
oof_program = np.array(oof_program)

print()
print(f'зведено по всіх {len(programs)} доменах:')
print(f'  macro-F1 {f1_score(oof_true, oof_pred, average="macro"):.4f}')
print(f'  accuracy {accuracy_score(oof_true, oof_pred):.4f}')
print(f'для порівняння, рубіж «завжди збій»: '
      f'macro-F1 {f1_score(y_true, always_fail, average="macro"):.4f} · '
      f'accuracy {accuracy_score(y_true, always_fail):.4f}')

In [ ]:
# по класах окремо: де саме модель втрачає
precision, recall, f1_per_class, support = precision_recall_fscore_support(
    oof_true, oof_pred, labels=[FAIL, OK], zero_division=0)

for i, name in ((FAIL, 'збій '), (OK, 'успіх')):
    print(f'{name}  точність {precision[i]:.4f} · повнота {recall[i]:.4f} · '
          f'F1 {f1_per_class[i]:.4f} · прикладів {support[i]}')

## 11 · Розкладаємо якість по доменах — і натрапляємо на пастку

Тепер найцікавіше. Порахуємо macro-F1 окремо в кожній програмі й подивимось на
найгірші.

In [ ]:
domain_score = {}
domain_minor = {}      # скільки в домені прикладів МЕНШОГО класу
domain_size = {}

for program in programs:
    mask = oof_program == program
    if mask.sum() == 0:
        continue
    y_dom = oof_true[mask]
    n_ok = int(y_dom.sum())
    domain_score[program] = f1_score(y_dom, oof_pred[mask], average='macro')
    domain_minor[program] = min(n_ok, len(y_dom) - n_ok)
    domain_size[program] = len(y_dom)

worst_first = sorted(domain_score, key=lambda p: domain_score[p])
print('НАЙГІРШІ ВІСІМ ДОМЕНІВ')
print(f'{"домен":28s} {"рядків":>7s} {"меншого класу":>14s} {"macro-F1":>9s}')
for program in worst_first[:8]:
    print(f'{program:28s} {domain_size[program]:7d} {domain_minor[program]:14d} '
          f'{domain_score[program]:9.4f}')
print()
print('НАЙКРАЩІ ЧОТИРИ')
for program in worst_first[-4:]:
    print(f'{program:28s} {domain_size[program]:7d} {domain_minor[program]:14d} '
          f'{domain_score[program]:9.4f}')

Подивись на колонку «меншого класу». У найгірших доменах там **нуль, один або два**
приклади.

Ось що відбувається. Macro-F1 — це середнє двох чисел: F1 класу «збій» і F1 класу
«успіх». Якщо в домені лише **один** приклад «успіху» і модель його проґавила, F1
цього класу дорівнює нулю, а macro-F1 падає приблизно до половини — **незалежно від
того, як добре модель обробила решту сто сорок рядків**.

Симетрично працює й інший край: домен зі 120 збоями й одним успіхом, у якому модель
той один успіх угадала, дає рівно 1.0000. Не тому, що там усе ідеально, а тому, що
там **нічого міряти**.

Порахуймо це руками на конкретному домені.

In [ ]:
# Беремо ВЕЛИКИЙ домен, у якому меншого класу лише один-два приклади: саме на такому
# видно, як мало треба, щоб число обвалилось. Якщо таких немає — беремо найгірший.
candidates = [p for p in domain_score
              if domain_size[p] >= 50 and 1 <= domain_minor[p] <= 2]
program = min(candidates, key=lambda p: domain_score[p]) if candidates else worst_first[0]

mask = oof_program == program
y_dom, p_dom = oof_true[mask], oof_pred[mask]
n_fail = int((y_dom == FAIL).sum())
n_ok = int((y_dom == OK).sum())

print(f'домен {program}: збоїв {n_fail}, успіхів {n_ok}')
print()
print(f'F1 класу «збій»  = {f1_of_class(y_dom, p_dom, FAIL):.4f}')
print(f'F1 класу «успіх» = {f1_of_class(y_dom, p_dom, OK):.4f}')
print(f'macro-F1         = {macro_f1(y_dom, p_dom):.4f}')
print()
smaller = min(n_fail, n_ok)
print(f'Тобто це число майже цілком визначене долею {smaller} '
      f'{"прикладу" if smaller == 1 else "прикладів"} меншого класу — '
      f'при тому, що рядків у домені всього {len(y_dom)}.')

In [ ]:
# скільки взагалі доменів мають досить прикладів, щоб про них можна було щось сказати
print(f'{"меншого класу":>16s} {"доменів":>8s} {"бал від":>9s} {"до":>8s} {"середній":>9s}')
for low, high, label in ((0, 0, 'нуль'), (1, 4, '1..4'), (5, 19, '5..19'), (20, 10**9, '20 і більше')):
    group = [p for p in domain_score if low <= domain_minor[p] <= high]
    if not group:
        continue
    scores = [domain_score[p] for p in group]
    print(f'{label:>16s} {len(group):8d} {min(scores):9.4f} {max(scores):8.4f} '
          f'{statistics.mean(scores):9.4f}')

too_small = [p for p in domain_score if domain_minor[p] < 5]
print()
print(f'доменів, про які нема що сказати (меншого класу < 5): {len(too_small)} '
      f'із {len(domain_score)}')

**Висновок цього розділу.** Більшість зрізів надто малі, щоб мати думку. Їхні бали
розкидані по всій шкалі, і цей розкид не говорить про модель нічого — він говорить
про те, скільки прикладів меншого класу випадково опинилось у домені.

Тому далі беремо тільки ті домени, де меншого класу **щонайменше двадцять**.

In [ ]:
MIN_MINOR = 20
big_domains = sorted([p for p in domain_score
                      if domain_minor[p] >= MIN_MINOR and domain_size[p] >= 50],
                     key=lambda p: domain_score[p])

print(f'доменів, де є що міряти: {len(big_domains)}')
print()
print(f'{"домен":28s} {"рядків":>7s} {"меншого класу":>14s} {"macro-F1":>9s}')
for program in big_domains[:6]:
    print(f'{program:28s} {domain_size[program]:7d} {domain_minor[program]:14d} '
          f'{domain_score[program]:9.4f}')
print(f'{"…":28s}')
for program in big_domains[-3:]:
    print(f'{program:28s} {domain_size[program]:7d} {domain_minor[program]:14d} '
          f'{domain_score[program]:9.4f}')

worst_real = domain_score[big_domains[0]]
spread_real = statistics.pstdev([domain_score[p] for p in big_domains])
pooled = f1_score(oof_true, oof_pred, average='macro')
print()
print(f'зведено по всьому корпусу: {pooled:.4f}')
print(f'найгірший вимірюваний домен: {worst_real:.4f} ({big_domains[0]})')
print(f'розкид балів (σ): {spread_real:.4f}')

## 12 · Нульовий тест: а що дав би сам лише шум?

Тепер найтонше місце теми, і воно чисто математичне.

Уяви, що всі ці домени **однаково добрі**. У кожному лише сотня-дві прикладів, тому
заміряний бал — не сама якість, а її **оцінка** з випадковою похибкою. Один домен
випадково дасть трохи більше, інший трохи менше. Ми беремо **найменше** з багатьох
таких чисел — і воно буде помітно нижчим за правду. Не тому, що якийсь домен
гірший, а тому, що **мінімум із багатьох шумних оцінок зміщений униз**.

Отже правильне питання не «який найгірший домен», а: **чи цей найгірший домен
гірший, ніж вийшло б із самого лише шуму?**

Перевіряємо прямо. Беремо наші справжні передбачення й справжні мітки, але
**перемішуємо приписку рядків до доменів**. Кожен фіктивний домен отримує рівно
стільки ж рядків і рівно стільки ж прикладів кожного класу, скільки мав справжній,
— але рядки до нього набираються звідусіль. У такому світі домени не відрізняються
один від одного **за визначенням**. Повторюємо двісті разів.

In [ ]:
# Готуємо все, що не змінюється між повторами, ЗАЗДАЛЕГІДЬ: інакше двісті повторів
# перетворюються на десятки мільйонів зайвих порівнянь.
index_by_class = {c: np.where(oof_true == c)[0] for c in (FAIL, OK)}
domain_shape = {}                      # домен -> (скільки «збоїв», скільки «успіхів»)
for program in big_domains:
    mask = oof_program == program
    n_ok = int(oof_true[mask].sum())
    domain_shape[program] = (int(mask.sum()) - n_ok, n_ok)

rng = np.random.default_rng(0)
null_worst, null_spread = [], []
t0 = time.process_time()

for _ in range(200):
    # перемішуємо номери рядків усередині кожного класу й роздаємо їх доменам
    pool = {c: rng.permutation(index_by_class[c]) for c in (FAIL, OK)}
    taken = {FAIL: 0, OK: 0}
    scores = []
    for program in big_domains:
        need_fail, need_ok = domain_shape[program]
        picked = np.concatenate([pool[FAIL][taken[FAIL]:taken[FAIL] + need_fail],
                                 pool[OK][taken[OK]:taken[OK] + need_ok]])
        taken[FAIL] += need_fail
        taken[OK] += need_ok
        scores.append(macro_f1(oof_true[picked], oof_pred[picked]))
    null_worst.append(min(scores))
    null_spread.append(statistics.pstdev(scores))

null_worst.sort()
print(f'нульовий тест зайняв {time.process_time() - t0:.1f} с процесорного часу')
print()
print(f'справжній найгірший зріз:           {worst_real:.4f}')
print(f'нульовий найгірший, медіана:        {statistics.median(null_worst):.4f}')
print(f'нульовий найгірший, 5-й процентиль: {null_worst[int(0.05 * len(null_worst))]:.4f}')
print(f'нульовий найгірший, найнижче з 200: {null_worst[0]:.4f}')
print()
print(f'справжній розкид балів (σ): {spread_real:.4f}')
print(f'нульовий розкид, медіана:   {statistics.median(null_spread):.4f}')
print()
worse_by_chance = sum(1 for m in null_worst if m <= worst_real)
print(f'шум сам дав зріз не кращий за справжній найгірший: {worse_by_chance} разів із 200')

**Як це читати.** Різниця між зведеним числом і найгіршим доменом складається з
двох частин: тієї, що дає шум, і тієї, що дає справжня різниця між доменами. Нульовий
тест показує **першу**. Усе, що лишається понад неї, — друга.

І ще одна засторога, без якої цей тест не працює: він доводить щось про **дані**
лише тоді, коли модель спершу довела, що вміє на них працювати. Наша обігнала обидва
рубежі з розділу 5 із великим запасом. Якби вона ледве їх обганяла, «розкид по
доменах не відрізняється від шуму» означало б лише, що модель однаково погана скрізь.

## 13 · Якщо домени все-таки відрізняються — то чим?

Дві правдоподібні причини, і обидві можна порахувати.

**Зсув пріора** — у різних доменах різна частка меншого класу.
**Коваріатний зсув** — у різних доменах різна лексика, і частина її моделі незнайома.

Порахуємо кореляцію подоменного бала з кожною. Кореляція — це міра того, наскільки
два числа рухаються разом: `+1` означає «завжди разом», `0` — «ніяк не повʼязані»,
`−1` — «завжди в різні боки».

### Спершу заміряємо зсув пріора

Найпростіша річ, що різниться між доменами, — **частка класів**. Порахуємо її для
кожної програми, у якій хоча б 50 рядків.

In [ ]:
ok_share = {}
for program in programs:
    n = sum(1 for r in rows if r[0] == program)
    if n >= 50:
        ok_share[program] = sum(1 for r in rows if r[0] == program and r[3] == OK) / n

ordered = sorted(ok_share, key=lambda p: ok_share[p])
values = [ok_share[p] for p in ordered]
print(f'доменів із >= 50 рядків: {len(ordered)}')
print(f'частка «успіху» в корпусі загалом: {counts[OK] / len(rows):.4f}')
print(f'мінімум: {ordered[0]} {values[0]:.4f}')
print(f'максимум: {ordered[-1]} {values[-1]:.4f}')
print(f'медіана: {statistics.median(values):.4f}')
print()
print('квантилі 10 / 25 / 50 / 75 / 90:',
      ' · '.join(f'{values[int(q * (len(values) - 1))]:.4f}'
                 for q in (0.10, 0.25, 0.50, 0.75, 0.90)))

In [ ]:
# частка меншого класу в домені
minor_share = {p: domain_minor[p] / domain_size[p] for p in big_domains}

# Новизна лексики: частка символьних триграм домену, яких модель у навчанні не бачила.
# Кожен фолд проходимо рівно двічі — спершу навчальні рядки, потім відкладені.
novelty_seen = collections.Counter()
novelty_unseen = collections.Counter()

for fold in folds:
    held_out = set(fold)
    train_trigrams = set()
    for r in rows:
        if r[0] in held_out:
            continue
        text = r[1].lower()
        for i in range(len(text) - 2):
            train_trigrams.add(text[i:i + 3])
    for r in rows:
        if r[0] not in held_out:
            continue
        text = r[1].lower()
        for i in range(len(text) - 2):
            novelty_seen[r[0]] += 1
            if text[i:i + 3] not in train_trigrams:
                novelty_unseen[r[0]] += 1

novelty = {p: novelty_unseen[p] / novelty_seen[p]
           for p in novelty_seen if novelty_seen[p]}

def correlation(xs, ys):
    """Кореляція Пірсона, написана вручну — щоб було видно, з чого вона складається."""
    mx, my = statistics.mean(xs), statistics.mean(ys)
    top = sum((a - mx) * (b - my) for a, b in zip(xs, ys))
    bottom = (sum((a - mx) ** 2 for a in xs) * sum((b - my) ** 2 for b in ys)) ** 0.5
    return top / bottom

scores = [domain_score[p] for p in big_domains]
print(f'кореляція «частка меншого класу» × «бал»: '
      f'{correlation([minor_share[p] for p in big_domains], scores):+.4f}')
print(f'кореляція «новизна лексики»       × «бал»: '
      f'{correlation([novelty[p] for p in big_domains], scores):+.4f}')
print()
nv = sorted(big_domains, key=lambda p: novelty[p])
print('найнижча новизна лексики:', ', '.join(f'{p} {novelty[p]:.4f}' for p in nv[:3]))
print('найвища новизна лексики: ', ', '.join(f'{p} {novelty[p]:.4f}' for p in nv[-3:]))

Зверни увагу, який звʼязок вийшов сильнішим. Якщо це **частка меншого класу**, то
навіть та частина розкиду, що пережила нульовий тест, розповідає не про те, що домен
«важчий», а про те, як улаштована метрика на його складі класів.

## 14 · Поріг — окреме рішення, а не частина моделі

Класифікатор не видає мітку. Він видає **число** — оцінку ймовірності. Мітка
зʼявляється тоді, коли хтось порівняв це число з порогом. За замовчуванням поріг
дорівнює 0.5, і саме тому про нього забувають: він виглядає частиною моделі.

Пройдімо всі пороги від 0.05 до 0.95 і подивімось, скільки коштує це «замовчування».

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.05)
print(f'{"поріг":>6s} {"macro-F1":>9s} {"точність «успіх»":>17s} {"повнота «успіх»":>16s}')
best = None
for threshold in thresholds:
    predicted = (oof_prob >= threshold).astype(int)
    macro = f1_score(oof_true, predicted, average='macro')
    p, r, _, _ = precision_recall_fscore_support(oof_true, predicted, labels=[OK],
                                                 zero_division=0)
    print(f'{threshold:6.2f} {macro:9.4f} {p[0]:17.4f} {r[0]:16.4f}')
    if best is None or macro > best[1]:
        best = (threshold, macro)

at_default = f1_score(oof_true, (oof_prob >= 0.5).astype(int), average='macro')
print()
print(f'поріг за замовчуванням 0.50: macro-F1 {at_default:.4f}')
print(f'найкращий поріг {best[0]:.2f}:        macro-F1 {best[1]:.4f}')
print(f'різниця: {best[1] - at_default:.4f}')

Максимум кривої стоїть **усередині** перебраного діапазону, а не скраю — отже його
справді знайдено, а не вперто в межу перебору. Якби максимум опинився на краю,
діапазон треба було б розширити, перш ніж називати це число оптимумом.

## 15 · Свій поріг у кожному домені

І ще одна річ, що прямо звʼязує поріг зі зсувом домену. Якщо в новому домені інша
частка класів — а ми щойно бачили, що це так, — то й найкращий поріг у ньому інший.
Це найдешевша поправка на зсув із усіх відомих: **нових міток не потрібно взагалі**.

In [ ]:
common_threshold = best[0]
with_common, with_own, own_thresholds = [], [], []

for program in big_domains:
    mask = oof_program == program
    y_dom, prob_dom = oof_true[mask], oof_prob[mask]
    with_common.append(f1_score(y_dom, (prob_dom >= common_threshold).astype(int),
                                average='macro'))
    scored = [(f1_score(y_dom, (prob_dom >= t).astype(int), average='macro'), t)
              for t in thresholds]
    top = max(scored)
    with_own.append(top[0])
    own_thresholds.append(top[1])

print(f'спільний поріг {common_threshold:.2f} на всі домени: '
      f'середній бал {statistics.mean(with_common):.4f}')
print(f'кожному домену свій поріг:          середній бал {statistics.mean(with_own):.4f}')
print(f'різниця: {statistics.mean(with_own) - statistics.mean(with_common):.4f}')
print()
print(f'свої пороги: від {min(own_thresholds):.2f} до {max(own_thresholds):.2f}, '
      f'медіана {statistics.median(own_thresholds):.2f}')

⚠️ **Одна засторога, без якої це число фальшиве.** Ми дібрали поріг **на тих самих**
рядках, на яких потім звітуємо. Це підглядання у відповідь: частина виграшу — це не
поправка на зсув, а підгонка. Правильно робити так: різати домен навпіл, добирати
поріг на одній половині, звітувати на другій. Ми цього тут навмисне не зробили, щоб
показати різницю в чистому вигляді, — і саме тому число вище є **верхньою** оцінкою
виграшу, а не очікуваним.

In [ ]:
print(f'усього процесорного часу на зошит: {time.process_time() - START_CPU:.1f} с')
print()
print('Головне, що варто винести:')
print(' 1. accuracy на незбалансованих даних міряє дисбаланс, а не модель;')
print(' 2. випадкове й доменне розбиття відповідають на РІЗНІ питання;')
print(' 3. більшість зрізів надто малі, щоб мати думку — їх треба відкинути,')
print('    а не звітувати;')
print(' 4. найгірший зріз порівнюють із нулем, а не з середнім;')
print(' 5. поріг буває дорожчим за вибір моделі, і коштує він нічого.')

---

## Завдання трьох рівнів

### 🟢 Рівень 1
Зміни `MIN_MINOR` з 20 на 5, 10, 30 і 50. Побудуй таблицю «мінімум прикладів
меншого класу → скільки доменів лишилось → найгірший бал». **Зроблено, якщо** ти
можеш назвати число, починаючи з якого найгірший бал перестає різко падати, і
пояснити, чому саме там.

### 🟡 Рівень 2
Полагодь дірку в правилі розмітки: додай пропущені заперечення (`could not`,
`is not`, `does not`, `without`) до `FAIL_WORDS` і перезапусти зошит. **Зроблено,
якщо** ти назвав, як змінились три числа — розмір меншого класу, зведена macro-F1 і
найгірший вимірюваний домен, — і сказав, яке з них змінилось найсильніше й чому.

### 🔴 Рівень 3
Зроби добір порога чесним: розділи кожен великий домен навпіл, добери поріг на
першій половині, зміряй на другій. **Зроблено, якщо** ти назвав, скільки з виграшу
0.0175 (або скільки вийшло в тебе) пережило чесний поділ, і пояснив, куди поділась
решта.

### Підказки

- У рівні 1 дивись не тільки на найгірший бал, а й на те, скільки доменів лишилось:
  число, при якому доменів стає менше десятка, теж нічого не варте — тільки з
  протилежної причини.
- У рівні 2 найсильніше зміниться те число, у знаменнику якого стоїть менший клас.
- У рівні 3 половини домену треба брати **випадково**, а не «перші й другі»: рядки
  в каталозі йдуть у порядку, і перша половина може виявитись іншим підмножиною тем.